# Questão 7 - Previsão de demanda

***Objetivo:*** Desenvolver um modelo preditivo que diga exatamente quantas unidades venderemos no próximo mês para ajustar as compras com fornecedores.

Premissas obrigatórias:

- O período de treino deve incluir dados até 31/12/2023.
- O período de teste deve ser todo o mês de Janeiro de 2024.
- A previsão deve ser feita em base diária.
- Não é permitido utilizar dados futuros no treino (data leakage).
- Considere apenas o produto: "Motor de Popa Yamaha Evo Dash 155HP"

***Metodologia***: 
1. Utilize o dataset vendas_2023_2024.csv
2. Construa um modelo baseline simples, utilizando: Média móvel dos últimos 7 dias de vendas (considerando apenas dados anteriores à data prevista).
3. Gere a previsão diária de vendas para Janeiro de 2024.
4. Compare as previsões com os valores reais do período de teste utilizando a métrica: MAE — Mean Absolute Error
5. Responda objetivamente:
     a. O baseline é adequado para esse produto?
     b. Cite uma limitação desse método.

### 1. Carregamento e Identificação (Baseado no nome)

In [1]:
#importando bibliotecas
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error

In [2]:
# 1. Carregar e padronizar datas (Resolvendo o caos dos formatos)
df_vendas = pd.read_csv('datasets/vendas_2023_2024.csv')
df_produtos = pd.read_csv('datasets/produtos_raw.csv')

### 2. Tratamento de Datas e Filtragem

In [3]:
# Identificação robusta pelo nome
nome_alvo = "Motor de Popa Yamaha Evo Dash 155HP"
# Pegamos o primeiro valor  para garantir que seja um número (escalar) e não uma lista
id_produto = df_produtos.loc[df_produtos['name'] == nome_alvo, 'code'].values
print(f"ID identificado para o produto: {id_produto}") # Deve imprimir 54

ID identificado para o produto: [54]


In [4]:
# O dataset tem datas como '2023-09-10' e '15-09-2024' [2]
# Usar format='mixed' resolve o erro de parsing de diferentes padrões
df_vendas['sale_date'] = pd.to_datetime(df_vendas['sale_date'], dayfirst=False, errors='coerce', format='mixed')

# Filtragem do produto e agrupamento diário
df_motores = df_vendas[df_vendas['id_product'] == id_produto[0]].copy()
df_diario = df_motores.groupby('sale_date')['qtd'].sum().reset_index()

id_produto[0]

54

In [5]:
df_motores.head()

,Unnamed: 0.1,Unnamed: 0,id,id_client,id_product,qtd,total,sale_date,sale_date_dt
46,46,46,48,13,54,15,1823022.00,2024-05-30,2024-05-30
53,53,53,55,35,54,3,346373.80,2024-11-24,2024-11-24
71,71,71,74,45,54,11,1270038.85,2024-09-25,2024-09-25
442,442,442,451,42,54,13,1500955.35,2024-02-19,2024-02-19
494,494,494,503,45,54,11,1270038.85,2024-11-27,2024-11-27


### 3. Criação da Janela de tempo completa

In [6]:
# Agrupamento diário e preenchimento de lacunas (Calendário)
# É essencial incluir dias com zero vendas para uma média real [4, 5]
df_diario = df_motores.groupby('sale_date')['qtd'].sum().reset_index()
calendario = pd.date_range(start='2023-01-01', end='2024-01-31')
df_completo = pd.DataFrame({'sale_date': calendario}).merge(df_diario, on='sale_date', how='left').fillna(0)


In [7]:
# Construção do Baseline (Média Móvel de 7 dias)
# O shift(1) garante que a previsão para o dia 'T' use apenas dados de 'T-1' para trás (evita data leakage)
df_completo['forecast'] = df_completo['qtd'].rolling(window=7).mean().shift(1)

### 4. Avaliação e Validação (Questão 7.2)

In [8]:
# Avaliação do Modelo (Janeiro de 2024)
teste_jan = df_completo[(df_completo['sale_date'] >= '2024-01-01') & 
                        (df_completo['sale_date'] <= '2024-01-31')].copy()

In [9]:
# Cálculo do MAE
mae = mean_absolute_error(teste_jan['qtd'], teste_jan['forecast'])

In [10]:
#Resultados para Validação
soma_semana_1 = teste_jan.head(7)['forecast'].sum()
print("-" * 30)
print(f"Previsão total (01/01 a 07/01): {round(soma_semana_1)}")
print(f"MAE para Janeiro/2024: {mae:.4f}")

------------------------------
Previsão total (01/01 a 07/01): 0
MAE para Janeiro/2024: 1.6406


### 5. Considerações

***Validação***: A soma total da previsão de vendas (arredondada) para a primeira semana de Janeiro de 2024 é 0. Isso ocorre porque não houve vendas registradas para o ID 54 nos últimos 7 dias de dezembro de 2023 no histórico fornecido

***Como o baseline foi construído?*** Através de uma Média Móvel Simples (SMA) de 7 dias. O modelo assume que a demanda de amanhã será a média do que foi vendido na última semana.

***Como evitou data leakage?*** Utilizando a função .shift(1) no pandas. Isso impede que o valor real de vendas do dia da previsão seja incluído no cálculo da sua própria média, garantindo que usemos apenas o passado para prever o futuro.

***Limitação do modelo:*** O atraso temporal (lag). Como o modelo é uma média, ele demora a reagir a picos de demanda. Se o produto começar a vender muito hoje, a média só subirá significativamente daqui a alguns dias, deixando o estoque do Sr. Almir desatualizado no momento crítico. Além disso, para produtos de demanda intermitente (como motores caros que não vendem todo dia), a média móvel gera valores fracionados que não representam bem a realidade de "venda ou não venda"
.